In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, RoFormerConfig, RoFormerModel
from torch.optim import AdamW
from datasets import load_dataset, load_from_disk
from tqdm import tqdm
from sklearn.metrics import f1_score

b:\stacks\Machine_learning_projects\Video_Summarization_Translation\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 1. Initialize Tokenizer and Model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class MiniSeg(nn.Module):
    def __init__(self, sentence_model_name=model_name):
        super(MiniSeg, self).__init__()
        self.sentence_encoder = AutoModel.from_pretrained(sentence_model_name)
        config = RoFormerConfig(
            num_hidden_layers=12,
            num_attention_heads=8,
            hidden_size=384,
            intermediate_size=1536,
            max_position_embeddings=2048
        )
        self.document_encoder = RoFormerModel(config)
        self.classifier = nn.Linear(384, 1)

    def forward(self, input_ids, attention_mask, grad_sample=0.5):
        # Gradient Sampling as per paper: reduce memory by backpropping 50% of docs 
        use_grads = (torch.rand(1).item() < grad_sample) if self.training else True
        
        with torch.set_grad_enabled(use_grads):
            outputs = self.sentence_encoder(input_ids=input_ids, attention_mask=attention_mask)
            sentence_embeddings = outputs.last_hidden_state.mean(dim=1).unsqueeze(0) 
            
        doc_outputs = self.document_encoder(inputs_embeds=sentence_embeddings)
        return self.classifier(doc_outputs.last_hidden_state).squeeze(-1)

# 2. Optimized Data Helper
def get_example_data(example, max_tokens=128, max_sentences=512):
    """
    Truncates sentences and document length to save VRAM.
    YTSEG documents average 196.2 sentences but can be much longer.
    """
    sentences = example['text_ref'][:max_sentences] # Truncate document length
    inputs = tokenizer(
        sentences, 
        padding=True, 
        truncation=True, 
        return_tensors="pt", 
        max_length=max_tokens # Truncate sentence length
    )
    
    label_str = example['target_binary_ref'].replace("|=", "")
    labels = torch.tensor([int(c) for c in label_str[:max_sentences]]).float()
    return inputs['input_ids'], inputs['attention_mask'], labels

# 3. Load Dataset
dataset = load_from_disk("B:\\stacks\\Machine_learning_projects\\Video_Summarization_Translation\\datasets\\ytseg-dataset")
train_data = dataset['train']
val_data = dataset['validation']

# 4. Training Setup with AMP (Mixed Precision)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MiniSeg().to(device)
optimizer = AdamW(model.parameters(), lr=2.5e-5)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([10.0]).to(device))

# Scaler for Automatic Mixed Precision (AMP)
scaler = torch.amp.GradScaler('cuda')

# 5. Training Loop
for epoch in range(20):
    model.train()
    total_loss = 0
    train_pbar = tqdm(train_data, desc=f"Epoch {epoch+1}/15 [Train]")
    
    for example in train_pbar:
        # Step A: Get truncated data to save memory
        ids, mask, labels = get_example_data(example, max_tokens=128, max_sentences=512)
        ids, mask, labels = ids.to(device), mask.to(device), labels.to(device).unsqueeze(0)
        
        optimizer.zero_grad()
        
        # Step B: Use Autocast for FP16 (halves memory for activations)
        with torch.amp.autocast('cuda'):
            logits = model(ids, mask)
            min_len = min(logits.shape[1], labels.shape[1])
            loss = criterion(logits[:, :min_len], labels[:, :min_len])
        
        # Step C: Scale and Backprop
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
        # Step D: Clear Cache frequently
        del ids, mask, labels, logits, loss
        torch.cuda.empty_cache()

    # 6. Evaluation Loop
    model.eval()
    all_preds, all_labels = [], []
    val_pbar = tqdm(val_data, desc=f"Epoch {epoch+1}/15 [Val]")
    
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            for example in val_pbar:
                ids, mask, labels = get_example_data(example, max_tokens=64, max_sentences=128)
                ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
                
                logits = model(ids, mask).squeeze(0)
                preds = (torch.sigmoid(logits) > 0.5).int()
                
                min_len = min(len(preds), len(labels))
                all_preds.extend(preds[:min_len].cpu().numpy())
                all_labels.extend(labels[:min_len].cpu().numpy())

    epoch_f1 = f1_score(all_labels, all_preds)
    print(f"\n--- Epoch {epoch+1} Summary ---")
    print(f"Avg Training Loss: {total_loss/len(train_data):.4f}")
    print(f"Validation F1 Score: {epoch_f1:.4f}\n")
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(), # If using AMP
        'f1_score': epoch_f1,
    }
    torch.save(checkpoint, f"../models/miniseg/epoch_{epoch+1}.pt")

Epoch 1/15 [Val]: 100%|██████████| 1447/1447 [00:37<00:00, 38.32it/s]



--- Epoch 1 Summary ---
Avg Training Loss: 0.8712
Validation F1 Score: 0.3821



Epoch 2/15 [Val]: 100%|██████████| 1447/1447 [00:38<00:00, 37.94it/s]



--- Epoch 2 Summary ---
Avg Training Loss: 0.7453
Validation F1 Score: 0.3995



Epoch 3/15 [Val]: 100%|██████████| 1447/1447 [00:37<00:00, 38.93it/s]



--- Epoch 3 Summary ---
Avg Training Loss: 0.6919
Validation F1 Score: 0.4001



... (19 outputs left out)


--- Epoch 13 Summary ---
Avg Training Loss: 0.4113
Validation F1 Score: 0.4131



Epoch 14/15 [Val]: 100%|██████████| 1447/1447 [00:36<00:00, 39.80it/s]



--- Epoch 14 Summary ---
Avg Training Loss: 0.3921
Validation F1 Score: 0.3937



Epoch 15/15 [Train]:  24%|██▍       | 3940/16404 [13:30<5:09:00,  1.49s/it, loss=0.7331]